In [7]:
import json
import re

import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from plotly.subplots import make_subplots

from benchmark_notebook import theme  # noqa: F401  registers thesis template
from benchmark_notebook.io import CloudStorage

In [8]:
attribution = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        key="scope-commit-attribution",
        version="1.0.0",
        location="main",
        partition="default",
    )
)
attribution.head()

name,branch,identifier,date,payload
str,str,str,str,str
"""lnls-fac/apsuite""","""master""","""495868aba47c80c19e56e30b59d010…","""2026-04-09T15:03:04Z""","""{""oid"": ""495868aba47c80c19e56e…"
"""lnls-fac/apsuite""","""master""","""629b55ca5c77ff781394c2fa69e224…","""2026-04-07T19:26:28Z""","""{""oid"": ""629b55ca5c77ff781394c…"
"""lnls-fac/apsuite""","""master""","""a3879f246b276649f85a0bb2c7d76b…","""2026-04-01T17:51:16Z""","""{""oid"": ""a3879f246b276649f85a0…"
"""lnls-fac/apsuite""","""master""","""e4cb1639aaef2ac2a4934127070ee6…","""2026-03-13T19:06:53Z""","""{""oid"": ""e4cb1639aaef2ac2a4934…"
"""lnls-fac/apsuite""","""master""","""3f1b5750db9b30e27b6c5f0e1525b9…","""2026-03-13T19:03:14Z""","""{""oid"": ""3f1b5750db9b30e27b6c5…"


In [9]:
cherry = re.compile(r"\(cherry picked from commit ([0-9a-f]{40})\)")


def classify(node, branch, repo):
    if node.get("messageBody") and cherry.search(node["messageBody"]):
        return "cherry_pick"
    prs = [
        p
        for p in (node.get("associatedPullRequests") or {}).get("nodes") or []
        if p.get("merged")
        and p.get("baseRefName") == branch
        and (p.get("repository") or {}).get("nameWithOwner") == repo
    ]
    if not prs:
        return "direct_push"
    if len(prs) == 1:
        return "pr_attributed"
    if (
        sum(1 for p in prs if (p.get("mergeCommit") or {}).get("oid") == node["oid"])
        == 1
    ):
        return "pr_attributed"
    return "ambiguous"


def label(row):
    return classify(json.loads(row["payload"]), row["branch"], row["name"])


classified = attribution.with_columns(
    pl.struct(["payload", "branch", "name"])
    .map_elements(label, return_dtype=pl.Utf8)
    .alias("class")
)

In [10]:
classes = ["direct_push", "pr_attributed", "cherry_pick", "ambiguous"]

pivot = (
    classified.group_by("name", "class")
    .len()
    .pivot(on="class", index="name", values="len")
)
missing = [c for c in classes if c not in pivot.columns]
pivot = pivot.with_columns(
    [pl.lit(0).cast(pl.UInt32).alias(c) for c in missing]
).fill_null(0)

rates = (
    pivot.with_columns(
        (pl.col("direct_push") + pl.col("pr_attributed")).alias("denominator"),
    )
    .with_columns(
        (pl.col("direct_push") / pl.col("denominator")).alias("rate"),
        (
            pl.col("direct_push")
            + pl.col("pr_attributed")
            + pl.col("cherry_pick")
            + pl.col("ambiguous")
        ).alias("commits"),
    )
    .filter(pl.col("denominator") > 0)
    .sort("rate", descending=True)
)
rates

name,direct_push,pr_attributed,cherry_pick,ambiguous,denominator,rate,commits
str,u32,u32,u32,u32,u32,f64,u32
"""bilby-dev/bilby""",1804,158,0,0,1962,0.91947,1962
"""idiap/coqui-ai-TTS""",1326,130,0,0,1456,0.910714,1456
"""valkey-io/valkey-py""",1615,162,0,0,1777,0.908835,1777
"""xenserver/host-installer""",2162,245,4,0,2407,0.898214,2411
"""quantumlib/Cirq""",3660,1055,0,0,4715,0.776246,4715
…,…,…,…,…,…,…,…
"""aws-deadline/deadline-cloud""",2,814,0,0,816,0.002451,816
"""shimming-toolbox/shimming-tool…",1,420,0,0,421,0.002375,421
"""Benchling-Open-Source/allotrop…",2,1005,0,0,1007,0.001986,1007


In [11]:
palette = pio.templates["thesis"].layout.colorway

(
    make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.12,
    )
    .add_trace(
        go.Histogram(
            x=rates["rate"],
            xbins=dict(start=0, end=1, size=0.05),
            marker=dict(color=palette[3], line=dict(width=0)),
            showlegend=False,
        ),
        row=1,
        col=1,
    )
    .add_trace(
        go.Scatter(
            x=rates["commits"],
            y=rates["rate"],
            mode="markers",
            marker=dict(size=4, color=palette[3]),
            text=rates["name"],
            hovertemplate="%{text}<br>commits=%{x}<br>rate=%{y:.3f}<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    .update_xaxes(title_text="Direct Push Rate", range=[0, 1], row=1, col=1)
    .update_yaxes(title_text="Repositories", row=1, col=1)
    .update_xaxes(title_text="Trunk Commits", type="log", row=1, col=2)
    .update_yaxes(title_text="Direct Push Rate", range=[0, 1], row=1, col=2)
    .update_layout(width=600, height=300, margin=dict(l=70, r=30, t=40, b=60))
    .show(config={"toImageButtonOptions": {"format": "svg"}})
)

In [12]:
with pl.Config(tbl_rows=20, fmt_str_lengths=80):
    print("Top 10 direct push rates:")
    print(
        rates.select("name", "rate", "commits", "direct_push", "pr_attributed").head(10)
    )
    print("\nBottom 10 direct push rates:")
    print(
        rates.select("name", "rate", "commits", "direct_push", "pr_attributed").tail(10)
    )

Top 10 direct push rates:
shape: (10, 5)
┌───────────────────────────────────────┬──────────┬─────────┬─────────────┬───────────────┐
│ name                                  ┆ rate     ┆ commits ┆ direct_push ┆ pr_attributed │
│ ---                                   ┆ ---      ┆ ---     ┆ ---         ┆ ---           │
│ str                                   ┆ f64      ┆ u32     ┆ u32         ┆ u32           │
╞═══════════════════════════════════════╪══════════╪═════════╪═════════════╪═══════════════╡
│ bilby-dev/bilby                       ┆ 0.91947  ┆ 1962    ┆ 1804        ┆ 158           │
│ idiap/coqui-ai-TTS                    ┆ 0.910714 ┆ 1456    ┆ 1326        ┆ 130           │
│ valkey-io/valkey-py                   ┆ 0.908835 ┆ 1777    ┆ 1615        ┆ 162           │
│ xenserver/host-installer              ┆ 0.898214 ┆ 2411    ┆ 2162        ┆ 245           │
│ quantumlib/Cirq                       ┆ 0.776246 ┆ 4715    ┆ 3660        ┆ 1055          │
│ XRPLF/xrpl-py              